In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import scipy.stats as stats

In [3]:
df = pd.read_csv("/Users/josiah/DTSC 2301/Flight Data/12683453/flights.csv")

/var/folders/yv/7g7fhntn0bddq69knwx72gj80000gn/T/ipykernel_20972/2435915714.py:1: DtypeWarning: Columns (0: altitude) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/josiah/DTSC 2301/Flight Data/12683453/flights.csv")


In [6]:
df = df[df["route"] == "R1"]
df["altitude"] = pd.to_numeric(df["altitude"])
print(df["flight"].nunique(), "flights")

182 flights


In [ ]:
df = df.sort_values(["flight", "time"])

# Power (watts) at each reading
df["power_W"] = df["battery_voltage"] * df["battery_current"]

# Seconds between readings within each flight
df["dt"] = df.groupby("flight")["time"].diff().fillna(0)

# Energy per reading (joules). battery voltage x battery current x time step (for each sensor reading)
df["energy_J"] = df["power_W"] * df["dt"]

# Collapse to one row per flight. The data is measured in seconds so we divide energy_j by 3600 to get watt hours
flights = df.groupby("flight").agg(
    energy_Wh=("energy_J", lambda x: x.sum() / 3600),
    speed=("speed", "first"),
    payload=("payload", "first"),
    altitude=("altitude", "first"),
    mean_wind=("wind_speed", "mean"),
).reset_index()

flights.head()

,flight,energy_Wh,speed,payload,altitude,mean_wind
0,8,21.933690,4.0,0.0,25,2.914185
1,10,16.528655,6.0,0.0,25,5.111714
2,12,13.011607,10.0,0.0,25,5.998581
3,14,14.199480,12.0,250.0,25,6.228249
4,15,14.239125,10.0,250.0,25,5.532143


In [8]:
flights.describe()

,flight,energy_Wh,speed,payload,altitude,mean_wind
count,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000
mean,159.120879,21.200278,8.076923,259.615385,62.500000,4.533689
std,65.770543,4.814393,2.831281,207.533171,27.904484,0.848055
min,8.000000,13.011607,4.000000,0.000000,25.000000,2.566423
25%,112.250000,17.554381,6.000000,0.000000,50.000000,3.934096
50%,159.500000,20.801324,8.000000,250.000000,62.500000,4.396080
75%,205.750000,24.580899,10.000000,500.000000,75.000000,5.100019
max,277.000000,36.225395,12.000000,750.000000,100.000000,6.670192
